# Model Training & Evaluation for Fraud Detection

This notebook demonstrates robust model training and evaluation using engineered features. Models include Logistic Regression, XGBoost, and LightGBM, with advanced handling of class imbalance, hyperparameter optimization, and thorough metric reporting.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import os
import sqlite3
import joblib
import logging
import sys
import time
import json
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb

# Try to import SHAP, make it optional for now
try:
    import shap
    SHAP_AVAILABLE = True
    print(" SHAP imported successfully")
except ImportError:
    SHAP_AVAILABLE = False
    print(" SHAP not available - will use alternative explainability")

# Import alternative explainability libraries
try:
    import eli5
    from eli5.sklearn import PermutationImportance
    ELI5_AVAILABLE = True
    print(" ELI5 imported successfully")
except ImportError:
    ELI5_AVAILABLE = False
    print(" ELI5 not available")

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve, auc, roc_curve
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE

print(" All core libraries imported successfully!")

# Set up explainability status
if SHAP_AVAILABLE:
    EXPLAINABILITY_METHOD = "SHAP"
elif ELI5_AVAILABLE:
    EXPLAINABILITY_METHOD = "ELI5"
else:
    EXPLAINABILITY_METHOD = "Basic Feature Importance"
    
print(f" Explainability method: {EXPLAINABILITY_METHOD}")

In [ ]:
# Configuration Management
CONFIG = {
    "features_csv": '../data/features/paysim_features.csv',
    "features_sqlite": '../data/features/features.db',
    "models_dir": '../backend/models',
    "log_file": "model_training.log",
    "test_size": 0.2,
    "validation_size": 0.1, # Proportion of full training set for validation
    "split_random_state": 42,
    "smote_random_state": 42,
    "lr_random_state": 42,
    "xgb_random_state": 42,
    "lgb_random_state": 42,
    "tuning_sample_size": 500000,
    "tuning_cv_folds": 3,
    "tuning_n_iter": 25,
    "tuning_random_state": 42,
    "xgb_param_grid": {
        'n_estimators': [100, 200],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    },
    "early_stopping_rounds": 20,
    "final_model_random_state": 42
}

In [ ]:
# Setup Centralized Logging
log_format = '%(asctime)s - %(levelname)s - %(message)s'
# Remove existing handlers if re-running cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO, format=log_format,
                    handlers=[logging.FileHandler(CONFIG["log_file"]), 
                              logging.StreamHandler(sys.stdout)])

logging.info("Logging initialized.")
logging.info(f"Configuration loaded: {json.dumps(CONFIG, indent=2)}")
# Suppress specific warnings from libraries if needed
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.utils.validation')
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost.training')

## Load Engineered Features
Load features from previous notebook.

In [ ]:
# Try loading PaySim features, fallback to SQLite if CSV not found
features_csv = CONFIG["features_csv"]
features_sqlite = CONFIG["features_sqlite"]
df = None # Initialize df
if os.path.exists(features_csv):
    logging.info(f"Loading features from CSV: {features_csv}")
    df = pd.read_csv(features_csv)
elif os.path.exists(features_sqlite):
    logging.info(f"CSV not found. Loading features from SQLite: {features_sqlite}")
    conn = sqlite3.connect(features_sqlite)
    df = pd.read_sql('SELECT * FROM paysim_features', conn)
    conn.close()
else:
    error_msg = 'No features file found. Please run the feature engineering notebook.'
    logging.error(error_msg)
    raise FileNotFoundError(error_msg)

logging.info(f"Loaded DataFrame shape: {df.shape}")
logging.info(f"DataFrame columns: {list(df.columns)}")

## Data Preprocessing
Select features, scale numeric features, and split data into training, validation, and test sets.

In [ ]:
target = 'isFraud'
# Select only numeric features for modeling (excluding target)
X = df.drop(columns=[target]).select_dtypes(include=[np.number])
y = df[target]
logging.info(f"Feature matrix X shape: {X.shape}")
logging.info(f"Target vector y shape: {y.shape}")

# Step 1: Split into Training (full) and Test sets
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, 
    test_size=CONFIG["test_size"], 
    stratify=y, 
    random_state=CONFIG["split_random_state"]
)
logging.info(f"Initial split: Train_full={len(y_train_full)}, Test={len(y_test)}")

# Step 2: Split Training (full) into Training and Validation sets
val_size_abs = int(CONFIG["validation_size"] * len(X)) # Validation size relative to original data
train_size_full = len(X_train_full)
val_size_rel = val_size_abs / train_size_full if train_size_full > 0 else 0

if val_size_rel >= 1.0 or val_size_rel <= 0:
     logging.warning(f"Calculated relative validation size ({val_size_rel:.2f}) is invalid. Setting validation size to 10% of training data.")
     val_size_rel = 0.1 # Default fallback relative validation size

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, 
    test_size=val_size_rel, 
    stratify=y_train_full, 
    random_state=CONFIG["split_random_state"]
)
logging.info(f"Secondary split: Train={len(y_train)}, Validation={len(y_val)}")

# Step 3: Scale features - Fit ONLY on X_train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
logging.info("Features scaled using StandardScaler (fit on training data only).")

## Handle Class Imbalance
Apply SMOTE to the **training set only** to balance classes.

In [ ]:
smote = SMOTE(random_state=CONFIG["smote_random_state"])
logging.info(f"Applying SMOTE to the training set (Shape before: {X_train_scaled.shape}).")
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
logging.info(f"SMOTE applied. Resampled training set shape: {X_train_res.shape}")
logging.info(f"Class distribution after SMOTE: {np.bincount(y_train_res)}")

## Train Logistic Regression
Baseline model for comparison. Trained on resampled data.

In [ ]:
logging.info("Training Logistic Regression model...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=CONFIG["lr_random_state"])
lr.fit(X_train_res, y_train_res)
y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]
logging.info("Logistic Regression training complete.")

## Train XGBoost (Initial)
Gradient boosting model for high performance. Trained on resampled data.

In [ ]:
logging.info("Training initial XGBoost model...")
scale_pos_weight_initial = (len(y_train_res) - sum(y_train_res)) / sum(y_train_res) if sum(y_train_res) > 0 else 1
logging.info(f"Calculated scale_pos_weight for initial XGBoost: {scale_pos_weight_initial:.4f}")
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight_initial,
    n_estimators=200, 
    max_depth=6, 
    learning_rate=0.05, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    random_state=CONFIG["xgb_random_state"], 
    use_label_encoder=False, 
    eval_metric='logloss',
    n_jobs=-1
)
xgb_model.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_model.predict(X_test_scaled)
y_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
logging.info("Initial XGBoost training complete.")

## Train LightGBM
Efficient boosting model for large datasets. Trained on resampled data.

In [ ]:
logging.info("Training LightGBM model...")
lgb_model = lgb.LGBMClassifier(
    class_weight='balanced', 
    n_estimators=200, 
    max_depth=6, 
    learning_rate=0.05, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    random_state=CONFIG["lgb_random_state"],
    n_jobs=-1
)
lgb_model.fit(X_train_res, y_train_res)
y_pred_lgb = lgb_model.predict(X_test_scaled)
y_proba_lgb = lgb_model.predict_proba(X_test_scaled)[:, 1]
logging.info("LightGBM training complete.")

## Initial Model Evaluation
Evaluate all models with robust metrics on the **test set**.

In [ ]:
# Evaluation function
def evaluate_model(y_true, y_pred, y_proba, model_name):
    roc_auc = roc_auc_score(y_true, y_proba)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    precision_curve_vals, recall_curve_vals, _ = precision_recall_curve(y_true, y_proba)
    auc_pr_val = auc(recall_curve_vals, precision_curve_vals)
    
    logging.info(f"--- {model_name} Evaluation ---")
    logging.info(f'AUC-ROC: {roc_auc:.4f}')
    logging.info(f'Precision: {precision:.4f}')
    logging.info(f'Recall: {recall:.4f}')
    logging.info(f'F1 Score: {f1:.4f}')
    logging.info(f'AUC-PR: {auc_pr_val:.4f}')
    logging.info(f'Confusion Matrix:\n{cm}')
    return {'roc_auc': roc_auc, 'precision': precision, 'recall': recall, 'f1': f1, 'auc_pr': auc_pr_val}

In [ ]:
logging.info("Starting initial evaluation of models on the test set...")
metrics = {}
metrics['Logistic Regression'] = evaluate_model(y_test, y_pred_lr, y_proba_lr, 'Logistic Regression')
metrics['XGBoost'] = evaluate_model(y_test, y_pred_xgb, y_proba_xgb, 'XGBoost')
metrics['LightGBM'] = evaluate_model(y_test, y_pred_lgb, y_proba_lgb, 'LightGBM')
logging.info("Initial model evaluation complete.")

## Initial Model Selection

Based on the initial evaluation metrics, particularly AUC-PR and Recall which are crucial for imbalanced fraud detection, we select the model with the best performance for hyperparameter tuning. XGBoost typically performs strongly in these scenarios.

In [ ]:
# Log the model selection decision based on initial metrics
# Find the best model based on AUC-PR (adjust metric if needed)
best_initial_model = max(metrics, key=lambda k: metrics[k]['auc_pr'])
logging.info(f"Initial model selection: {best_initial_model} chosen for hyperparameter tuning based on highest initial AUC-PR ({metrics[best_initial_model]['auc_pr']:.4f}).")

## Save Initial Model & Scaler (Optional)
Save the initially trained XGBoost model and scaler artifact before tuning (useful for comparison or quick deployment).

In [ ]:
# Ensure models directory exists before saving
models_dir = CONFIG["models_dir"]
os.makedirs(models_dir, exist_ok=True)
joblib.dump(xgb_model, os.path.join(models_dir, 'xgb_fraud_model_initial.pkl'))
joblib.dump(scaler, os.path.join(models_dir, 'scaler_initial.pkl'))
logging.info(f"Initial XGBoost model and scaler saved to {models_dir}")

## Next Steps (Before API)
Proceed to hyperparameter tuning for the selected model.

## Hyperparameter Tuning & Cross-Validation (XGBoost)
Optimize model parameters and validate performance with cross-validation for robust results.

##  Optimization Philosophy
- **Use approximation where safe, precision where necessary.**
- **Sampling, random search, histogram trees, and fewer folds reduce runtime significantly while maintaining near-identical accuracy during tuning.**
- **Final full-dataset retraining ensures no performance loss in production.**

This approach balances efficiency with accuracy:
1. Sample large resampled datasets for hyperparameter tuning
2. Use RandomizedSearchCV instead of exhaustive grid search
3. Leverage histogram-based tree methods for speed
4. Use fewer CV folds during tuning
5. Retrain final model on full resampled dataset for production accuracy

## Step 1: Sample for Fast Tuning
Sample a subset of the **resampled training data** for efficient hyperparameter search.

In [ ]:
# Step 1: Sample for Fast Tuning
# Sample N rows (or less if dataset is smaller) for efficient hyperparameter search
sample_size = min(CONFIG["tuning_sample_size"], len(X_train_res))
# Ensure reproducibility of sampling
sample_indices = np.random.RandomState(CONFIG["tuning_random_state"]).choice(
    len(X_train_res), size=sample_size, replace=False
)

X_sample = X_train_res[sample_indices]
# Handle y_train_res being potentially a Series or ndarray after SMOTE
if isinstance(y_train_res, pd.Series):
    y_sample = y_train_res.iloc[sample_indices]
else:
    y_sample = y_train_res[sample_indices]

logging.info(f"Original resampled training data: {len(X_train_res):,} rows")
logging.info(f"Sampled for tuning: {len(X_sample):,} rows ({len(X_sample)/len(X_train_res)*100:.1f}%)")
logging.info(f"Sample shape: {X_sample.shape}")
logging.info(f"Class distribution in sample: {np.bincount(y_sample)}")

## Step 2: Define Parameter Search Space
Use a broad but reasonable parameter space for efficient exploration.

In [ ]:
# Step 2: Define Parameter Search Space
param_grid = CONFIG["xgb_param_grid"]

logging.info("Parameter search space defined:")
for param, values in param_grid.items():
    logging.info(f"  {param}: {values}")
total_combinations = np.prod([len(v) for v in param_grid.values()])
logging.info(f"Total combinations possible in grid: {total_combinations}")
logging.info(f"RandomizedSearchCV will test {CONFIG['tuning_n_iter']} random combinations")

## Step 3: Randomized Search with Cross-Validation
Use RandomizedSearchCV with histogram-based trees for efficient parameter exploration.

In [ ]:
# Step 3: Randomized Search with Cross-Validation Setup
# Use N-fold CV for speed during tuning
cv_folds = CONFIG["tuning_cv_folds"]
cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=CONFIG["tuning_random_state"])

# Calculate scale_pos_weight for the *sample* (since SMOTE made it 50/50)
# For balanced data after SMOTE, this should be close to 1
sample_pos_weight = (len(y_sample) - np.sum(y_sample)) / np.sum(y_sample) if np.sum(y_sample) > 0 else 1

# Create RandomizedSearchCV with histogram-based trees
n_iter = CONFIG["tuning_n_iter"]
random_search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        tree_method='hist',  # Use faster histogram-based algorithm
        scale_pos_weight=sample_pos_weight, # Use weight calculated for the sample
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=CONFIG["tuning_random_state"],
        n_jobs=-1  # Use all available CPU cores for fitting each model
    ),
    param_distributions=param_grid,
    n_iter=n_iter, # Number of random combinations to try
    scoring='roc_auc', # Metric to optimize
    cv=cv,
    n_jobs=-1, # Parallelize CV folds if possible (-1 uses all cores)
    verbose=1, # Show some progress (level 1 is less noisy than 2)
    random_state=CONFIG["tuning_random_state"]
)

logging.info(f"Starting RandomizedSearchCV on {len(X_sample):,} samples...")
logging.info(f"Using {cv.n_splits}-fold cross-validation.")
logging.info(f"Testing {n_iter} parameter combinations.")
logging.info(f"Sample scale_pos_weight: {sample_pos_weight:.4f}")

## Step 4: Run Search and Save Progress
Execute the hyperparameter search and save results.

In [ ]:
# Step 4: Run Search and Save Progress
models_dir = CONFIG["models_dir"]
os.makedirs(models_dir, exist_ok=True)

# Run the hyperparameter search
start_time = time.time()
random_search.fit(X_sample, y_sample)
search_time = time.time() - start_time

# Save the search object for later analysis
search_results_path = os.path.join(models_dir, 'xgb_random_search.pkl')
joblib.dump(random_search, search_results_path)
logging.info(f"RandomizedSearchCV object saved to {search_results_path}")

# Log results
logging.info(f"Randomized search completed in {search_time:.1f} seconds")
logging.info(f"Best parameters found: {random_search.best_params_}")
logging.info(f"Best cross-validation ROC-AUC score: {random_search.best_score_:.4f}")

## Step 5: Visualize Parameter Importance from Tuning
Create visualizations to understand which parameters performed best during the search.

In [ ]:
# Step 5: Visualize Parameter Importance
# Create DataFrame from search results
results_df = pd.DataFrame(random_search.cv_results_)

# Display top 10 parameter combinations via logging
logging.info("Top 10 parameter combinations from RandomizedSearch:")
top_results = results_df.nlargest(10, 'mean_test_score')[
    ['mean_test_score', 'std_test_score', 'param_max_depth', 
     'param_learning_rate', 'param_n_estimators', 'param_subsample', 'param_colsample_bytree']
]
logging.info(f"\n{top_results.to_string()}") # Log the dataframe string

# Create heatmap of max_depth vs learning_rate performance
try:
    plt.figure(figsize=(10, 6))
    pivot = results_df.pivot_table(
        index='param_max_depth', 
        columns='param_learning_rate', 
        values='mean_test_score',
        aggfunc='mean'
    )
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='viridis')
    plt.title('Mean ROC-AUC by max_depth and learning_rate (Tuning Sample)')
    plt.xlabel('Learning Rate')
    plt.ylabel('Max Depth')
    plt.tight_layout()
    plt.show()
except Exception as e:
    logging.warning(f"Could not generate heatmap: {e}")

# Parameter distribution analysis
try:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    # Ensure axes is treated as a 2D array even if it's 1D
    axes = np.atleast_2d(axes)
    params_to_plot = [p for p in results_df.columns if p.startswith('param_')]
    
    plot_count = 0
    for i, param in enumerate(params_to_plot):
        if len(results_df[param].unique()) > 1: # Only plot if more than one value was tested
            row, col = divmod(plot_count, 3)
            if row >= axes.shape[0] or col >= axes.shape[1]: continue # Avoid index error if too many params
            ax = axes[row, col]
            # Group by parameter value and calculate mean score
            param_scores = results_df.groupby(param)['mean_test_score'].mean().sort_index()
            param_scores.plot(kind='bar', ax=ax, rot=0)
            ax.set_title(f'{param.replace("param_", "").title()} Performance')
            ax.set_ylabel('Mean ROC-AUC')
            ax.set_xlabel(param.replace("param_", "").title())
            #ax.tick_params(axis='x', rotation=0)
            plot_count += 1
        
    # Remove empty subplots if necessary
    total_plots_needed = len([p for p in params_to_plot if len(results_df[p].unique()) > 1])
    for i in range(plot_count, axes.size):
         row, col = divmod(i, 3)
         if row < axes.shape[0] and col < axes.shape[1]: # Check bounds before removing
            fig.delaxes(axes[row, col])

    plt.suptitle('Mean ROC-AUC per Hyperparameter Value (Tuning Sample)')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
    plt.show()
except Exception as e:
     logging.warning(f"Could not generate parameter distribution plots: {e}")

## Step 6: Final Model Training on Full Resampled Dataset
Retrain the best model on the **complete resampled training dataset** (X\_train\_res, y\_train\_res) for production accuracy.

In [ ]:
# Step 6: Final Model Training on Full Dataset
logging.info(f"Training final model (Standard) on full resampled dataset ({len(X_train_res):,} rows)...")

# Get best parameters from search
best_params = random_search.best_params_
logging.info(f"Using best parameters from RandomizedSearch: {best_params}")

# Create final model with best parameters and additional optimizations
final_model_standard = xgb.XGBClassifier(
    **best_params,
    tree_method='hist',  # Use histogram-based for speed
    eval_metric='logloss',
    use_label_encoder=False, # Already specified in RandomizedSearch, but good practice
    n_jobs=-1,  # Use all CPU cores
    random_state=CONFIG["final_model_random_state"]
    # Note: scale_pos_weight is not typically needed here as data is balanced by SMOTE
    # If not using SMOTE, calculate and include scale_pos_weight based on the original y_train
)

# Train on full resampled dataset
start_time_final = time.time()
final_model_standard.fit(X_train_res, y_train_res)
final_training_time = time.time() - start_time_final

logging.info(f"Final model (Standard) training completed in {final_training_time:.1f} seconds")
logging.info(f"Model trained on {len(X_train_res):,} samples")

## Step 7: Early Stopping Implementation (Retraining)
Retrain using the best parameters but implement early stopping using the **validation set** (X\_val\_scaled, y\_val) to potentially prevent overfitting and find optimal number of estimators.

In [ ]:
# Step 7: Early Stopping Implementation (Retraining)
logging.info("Training final model with early stopping using the validation set...")

# Use the validation set created during preprocessing
eval_set = [(X_val_scaled, y_val)]

# Train model with early stopping
final_model_es = xgb.XGBClassifier(
    **best_params,
    tree_method='hist',
    eval_metric='logloss', # Logloss is a common metric for eval set
    use_label_encoder=False,
    n_jobs=-1,
    random_state=CONFIG["final_model_random_state"]
    # n_estimators from best_params serves as the maximum limit
)

# Fit with validation set and early stopping callback
start_time_es = time.time()
early_stopping_rounds = CONFIG["early_stopping_rounds"]

# Newer XGBoost versions might handle callbacks slightly differently
# We prioritize using 'early_stopping_rounds' directly if possible
try:
    final_model_es.fit(
        X_train_res, y_train_res,
        eval_set=eval_set,
        early_stopping_rounds=early_stopping_rounds,
        verbose=False # Set to True or a number to see progress
    )
    logging.info(f"Early stopping triggered. Best iteration: {final_model_es.best_iteration}")
    logging.info(f"Best validation score ({final_model_es.eval_metric}): {final_model_es.best_score:.4f}")
except TypeError as e:
    logging.warning(f"Direct early_stopping_rounds failed ('{e}'). Trying callback method...")
    try:
        # Fallback for versions expecting callbacks
        final_model_es.fit(
            X_train_res, y_train_res,
            eval_set=eval_set,
            callbacks=[xgb.callback.EarlyStopping(rounds=early_stopping_rounds, save_best=True)],
            verbose=False
        )
        logging.info(f"Early stopping (callback) triggered. Best iteration: {final_model_es.best_iteration}")
        logging.info(f"Best validation score ({final_model_es.eval_metric}): {final_model_es.best_score:.4f}")
    except Exception as final_e:
        logging.error(f"Early stopping failed entirely ('{final_e}'). Training without early stopping as fallback.")
        # If both fail, just fit normally (though this shouldn't happen with modern XGBoost)
        final_model_es = xgb.XGBClassifier(**best_params, tree_method='hist', eval_metric='logloss', use_label_encoder=False, n_jobs=-1, random_state=CONFIG["final_model_random_state"])
        final_model_es.fit(X_train_res, y_train_res)

es_training_time = time.time() - start_time_es
logging.info(f"Final model (Early Stopping) training completed in {es_training_time:.1f} seconds")

# Compare number of estimators used
logging.info("--- Estimator Comparison ---")
logging.info(f"Standard model used specified n_estimators: {final_model_standard.n_estimators}")
if hasattr(final_model_es, 'best_iteration') and final_model_es.best_iteration is not None:
    # Note: best_iteration is 0-indexed, so add 1 for number of trees
    logging.info(f"Early stopping model used optimal n_estimators: {final_model_es.best_iteration + 1}")
else:
    logging.warning(f"Early stopping did not trigger or info unavailable. Model used max n_estimators: {final_model_es.n_estimators}")

## Step 8: Comprehensive Final Model Evaluation
Evaluate both final models (Standard and Early Stopping) on the **unseen test set** using multiple metrics.

In [ ]:
# Step 8: Comprehensive Final Model Evaluation on Test Set
logging.info("Starting final evaluation of models on the test set...")

final_evaluation_results = {}
final_models_to_evaluate = {
    'Standard Model': final_model_standard,
    'Early Stopping Model': final_model_es
}

for model_name, model in final_models_to_evaluate.items():
    logging.info(f"\n--- Evaluating {model_name} on Test Set ---")
    
    # Predictions on the final, unseen test set
    preds_test = model.predict(X_test_scaled)
    probs_test = model.predict_proba(X_test_scaled)[:, 1]
    
    # Use the evaluation function (defined earlier)
    results = evaluate_model(y_test, preds_test, probs_test, model_name)
    
    # Store detailed results including predictions/probabilities if needed later
    final_evaluation_results[model_name] = results
    final_evaluation_results[model_name]['predictions'] = preds_test
    final_evaluation_results[model_name]['probabilities'] = probs_test

# Log comparison
logging.info("--- Final Model Comparison Summary (Test Set) ---")
comparison_df = pd.DataFrame(final_evaluation_results).T
logging.info(f"\n{comparison_df[['roc_auc', 'precision', 'recall', 'f1', 'auc_pr']].round(4)}")

## Final Model Selection for Persistence

Compare the performance of the 'Standard Model' (trained for fixed estimators on full resampled data) and the 'Early Stopping Model' (trained on full resampled data with validation-based early stopping) on the **test set**. Select the better performing one, or the standard one if performance is very similar, for deployment.

In [ ]:
# Choose the final model to persist
# Example logic: Prefer Early Stopping if its AUC-PR is better, otherwise Standard
auc_pr_standard = final_evaluation_results.get('Standard Model', {}).get('auc_pr', 0)
auc_pr_es = final_evaluation_results.get('Early Stopping Model', {}).get('auc_pr', 0)

if auc_pr_es > auc_pr_standard:
    chosen_model_name = 'Early Stopping Model'
    chosen_model = final_model_es
    logging.info(f"Final model selection: {chosen_model_name} selected (AUC-PR {auc_pr_es:.4f} > {auc_pr_standard:.4f}).")
else:
    chosen_model_name = 'Standard Model'
    chosen_model = final_model_standard
    logging.info(f"Final model selection: {chosen_model_name} selected (AUC-PR {auc_pr_standard:.4f} >= {auc_pr_es:.4f}).")

# Store chosen model info for persistence and metadata
final_best_model_name = chosen_model_name
final_best_model = chosen_model
final_best_metrics = final_evaluation_results[final_best_model_name]
final_best_probs_test = final_best_metrics['probabilities'] # Probabilities on test set

## Step 9: Calibration Curves and Performance Visualization (Final Model)
Visualize the **chosen final model's** calibration and performance characteristics on the **test set**.

In [ ]:
# Step 9: Calibration Curves and Performance Visualization (Final Chosen Model on Test Set)
logging.info(f"Generating performance visualizations for the chosen final model: {final_best_model_name}")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Use results from the chosen best model evaluated on the test set
roc_auc_final = final_best_metrics['roc_auc']
auc_pr_final = final_best_metrics['auc_pr']

# 1. Calibration curve for the final chosen model
prob_true_test, prob_pred_test = calibration_curve(y_test, final_best_probs_test, n_bins=10)
axes[0, 0].plot(prob_pred_test, prob_true_test, marker='o', linewidth=2, label=final_best_model_name)
axes[0, 0].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
axes[0, 0].set_xlabel('Mean Predicted Probability (Test Set)')
axes[0, 0].set_ylabel('Fraction of Positives (Test Set)')
axes[0, 0].set_title(f'Calibration Curve ({final_best_model_name})')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. ROC curve for the final chosen model
fpr_test, tpr_test, _ = roc_curve(y_test, final_best_probs_test)
axes[0, 1].plot(fpr_test, tpr_test, linewidth=2, 
               label=f'{final_best_model_name} (AUC = {roc_auc_final:.4f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0, 1].set_xlabel('False Positive Rate (Test Set)')
axes[0, 1].set_ylabel('True Positive Rate (Test Set)')
axes[0, 1].set_title(f'ROC Curve ({final_best_model_name})')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Precision-Recall curve for the final chosen model
precision_curve_test, recall_curve_test, _ = precision_recall_curve(y_test, final_best_probs_test)
axes[1, 0].plot(recall_curve_test, precision_curve_test, linewidth=2,
               label=f'{final_best_model_name} (AUC-PR = {auc_pr_final:.4f})')
axes[1, 0].set_xlabel('Recall (Test Set)')
axes[1, 0].set_ylabel('Precision (Test Set)')
axes[1, 0].set_title(f'Precision-Recall Curve ({final_best_model_name})')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Feature importance for the final chosen model
# Check if feature names are available (they should be from original X)
feature_names = X.columns.tolist() if hasattr(X, 'columns') else [f'feature_{i}' for i in range(X_test_scaled.shape[1])]
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': final_best_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

axes[1, 1].barh(feature_importance['feature'], feature_importance['importance'])
# axes[1, 1].set_yticks(range(len(feature_importance)))
# axes[1, 1].set_yticklabels(feature_importance['feature'])
axes[1, 1].invert_yaxis() # Display most important at top
axes[1, 1].set_xlabel('Feature Importance')
axes[1, 1].set_title(f'Top 15 Feature Importances ({final_best_model_name})')
axes[1, 1].grid(True, axis='x', alpha=0.3)

plt.suptitle(f'Final Model Performance Analysis ({final_best_model_name} on Test Set)')
plt.tight_layout(rect=[0, 0.03, 1, 0.96])
plt.show()

# Log feature importance
logging.info(f"\nTop 15 Most Important Features ({final_best_model_name}):")
logging.info(f"\n{feature_importance.to_string(index=False)}")

## Step 9b: Explainability (SHAP) & Drift Check (PSI)
Apply SHAP for explainability and calculate PSI to check potential drift between training and test prediction distributions.

In [ ]:
# SHAP explainability for the chosen final model on the test set
logging.info(f"Calculating SHAP values for {final_best_model_name} on the test set...")
try:
    explainer = shap.TreeExplainer(final_best_model)
    # Use a sample of the test set for SHAP summary plots if it's large
    shap_sample_size = min(1000, X_test_scaled.shape[0]) 
    shap_indices = np.random.choice(X_test_scaled.shape[0], shap_sample_size, replace=False)
    X_test_scaled_sample = X_test_scaled[shap_indices]
    # Create a DataFrame for plotting with feature names
    X_test_sample_df = pd.DataFrame(X_test_scaled_sample, columns=X.columns)
    
    shap_values = explainer.shap_values(X_test_scaled_sample)
    
    logging.info("Generating SHAP summary plots...")
    # Bar plot (mean absolute SHAP value)
    shap.summary_plot(shap_values, X_test_sample_df, plot_type='bar', show=False)
    plt.title(f'SHAP Feature Importance ({final_best_model_name})')
    plt.show()
    
    # Beeswarm plot (distribution of SHAP values)
    shap.summary_plot(shap_values, X_test_sample_df, show=False)
    plt.title(f'SHAP Value Distribution ({final_best_model_name})')
    plt.show()
    logging.info("SHAP analysis complete.")
except Exception as e:
    logging.error(f"SHAP analysis failed: {e}")

In [ ]:
# PSI calculation for drift detection (comparing train probabilities vs test probabilities)
logging.info("Calculating Population Stability Index (PSI)...")
def calculate_psi(expected, actual, buckets=10):
    # Ensure inputs are numpy arrays
    expected = np.asarray(expected)
    actual = np.asarray(actual)
    
    # Determine bins using the range of the expected distribution
    min_val = np.min(expected)
    max_val = np.max(expected)
    breakpoints = np.linspace(min_val, max_val, buckets + 1)
    
    # Add small epsilon to max_val for histogram edge case
    breakpoints[-1] = max_val + 1e-6

    # Calculate percentages for each bucket
    expected_percents = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_percents = np.histogram(actual, bins=breakpoints)[0] / len(actual)
    
    # Replace 0 percentages with a small value to avoid division by zero / log(0)
    epsilon = 1e-6
    expected_percents = np.where(expected_percents == 0, epsilon, expected_percents)
    actual_percents = np.where(actual_percents == 0, epsilon, actual_percents)
    
    # Calculate PSI value
    psi_value = np.sum((actual_percents - expected_percents) * np.log(actual_percents / expected_percents))
    return psi_value

# Generate predictions on the *resampled training set* using the final model
logging.info(f"Generating predictions on resampled training set for PSI calculation...")
y_train_pred_proba = final_best_model.predict_proba(X_train_res)[:, 1]

# Compare training predictions (expected) vs test predictions (actual)
try:
    psi = calculate_psi(y_train_pred_proba, final_best_probs_test) # final_best_probs_test are from test set
    logging.info(f'PSI (Train vs Test Predictions): {psi:.4f}')
    if psi < 0.1:
        logging.info('PSI indicates no significant population drift.')
    elif psi < 0.25:
        logging.warning('PSI indicates minor population drift.')
    else:
        logging.error('PSI indicates major population drift!')
except Exception as e:
    logging.error(f"PSI calculation failed: {e}")

## Step 10: Model Persistence and Metadata Generation
Save the **chosen final model**, scaler, tuning report, and comprehensive metadata for deployment and reproducibility.

In [ ]:
# Step 10: Model Persistence and Metadata Generation
models_dir = CONFIG["models_dir"]
os.makedirs(models_dir, exist_ok=True)

# Save the chosen final model and the scaler (fit on original training data)
final_model_path = os.path.join(models_dir, 'final_xgb_model.pkl')
scaler_path = os.path.join(models_dir, 'scaler.pkl')
tuning_report_path = os.path.join(models_dir, 'xgb_tuning_report.csv')
metadata_path = os.path.join(models_dir, 'model_metadata.json')
search_results_path = os.path.join(models_dir, 'xgb_random_search.pkl') # Already saved, just logging path

logging.info(f"Saving final chosen model ({final_best_model_name}) to {final_model_path}")
joblib.dump(final_best_model, final_model_path)
logging.info(f"Saving scaler to {scaler_path}")
joblib.dump(scaler, scaler_path)

# Save hyperparameter search results dataframe if not already done
if not os.path.exists(tuning_report_path):
    results_df.to_csv(tuning_report_path, index=False)
    logging.info(f"Hyperparameter tuning report saved to {tuning_report_path}")

# Create comprehensive model metadata using final test set metrics
final_metrics_persist = {k: float(v) for k, v in final_best_metrics.items() 
                         if isinstance(v, (int, float, np.number))}

model_metadata = {
    'model_type': 'XGBoost',
    'chosen_model': final_best_model_name,
    'model_path': os.path.basename(final_model_path),
    'scaler_path': os.path.basename(scaler_path),
    'training_date': datetime.now().isoformat(),
    'dataset_info': {
        'total_original_samples': len(df),
        'training_samples_before_smote': len(y_train),
        'training_samples_after_smote': len(y_train_res),
        'validation_samples': len(y_val),
        'test_samples': len(y_test),
        'features': len(X.columns),
        'original_class_distribution': y.value_counts().to_dict()
    },
    'hyperparameter_search': {
        'search_type': 'RandomizedSearchCV',
        'cv_folds': CONFIG["tuning_cv_folds"],
        'n_iterations': CONFIG["tuning_n_iter"],
        'search_time_seconds': search_time,
        'best_params_found': best_params,
        'best_cv_score': float(random_search.best_score_)
    },
    'final_model_performance_on_test': final_metrics_persist,
    'final_model_training_time_seconds': final_training_time if final_best_model_name == 'Standard Model' else es_training_time,
    'feature_columns': X.columns.tolist(),
    'top_features': feature_importance.head(10).to_dict('records'),
    'psi_train_vs_test': psi if 'psi' in locals() else None # Include PSI if calculated
}

# Save metadata
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)
logging.info(f"Model metadata saved to {metadata_path}")

logging.info(" Model persistence and metadata generation completed successfully!")
logging.info(f" Files generated/saved in {models_dir}:")
logging.info(f"   - {os.path.basename(final_model_path)}")
logging.info(f"   - {os.path.basename(scaler_path)}") 
logging.info(f"   - {os.path.basename(tuning_report_path)}")
logging.info(f"   - {os.path.basename(metadata_path)}")
logging.info(f"   - {os.path.basename(search_results_path)}")

logging.info(f"\n Final Chosen Model ({final_best_model_name}) Summary (Test Set):")
logging.info(f"   ROC-AUC: {final_best_metrics['roc_auc']:.4f}")
logging.info(f"   Precision: {final_best_metrics['precision']:.4f}")
logging.info(f"   Recall: {final_best_metrics['recall']:.4f}")
logging.info(f"   F1-Score: {final_best_metrics['f1']:.4f}")
logging.info(f"   AUC-PR: {final_best_metrics['auc_pr']:.4f}")

logging.info(f"\n Artifacts ready for API deployment!")

## Final Workflow Summary
The notebook now executes the following optimized workflow:
1. **Setup:** Load libraries, configuration, and initialize logging.
2. **Data Handling:** Load features, split into Train/Validation/Test sets, scale features (fitting scaler only on Train).
3. **Imbalance Handling:** Apply SMOTE only to the Train set.
4. **Initial Modeling:** Train LR, XGB, LGB on resampled Train data.
5. **Initial Evaluation:** Evaluate models on the Test set, select best (XGBoost) for tuning.
6. **Hyperparameter Tuning (XGBoost):**
    - Sample resampled Train data.
    - Run RandomizedSearchCV using the sample.
    - Visualize tuning results.
7. **Final Model Training:**
    - Train 'Standard Model' on full resampled Train data using best params.
    - Train 'Early Stopping Model' on full resampled Train data using best params and the Validation set for stopping.
8. **Final Evaluation & Selection:** Evaluate both final models on the Test set, select the definitive 'best' model.
9. **Analysis & Visualization:** Generate calibration, ROC, PR curves, and feature importance plots for the chosen final model on the Test set.
10. **Explainability & Drift Check:** Apply SHAP and calculate PSI for the chosen final model.
11. **Persistence:** Save the chosen final model, scaler, tuning report, and comprehensive metadata JSON.

The process prioritizes efficiency during tuning and robustness through proper validation/testing and final retraining.

In [ ]:
# Note: The comprehensive optimization implementation is now integrated above (Steps 1-10).
# Placeholder for any future ad-hoc analysis if needed.

In [ ]:
# Example Placeholder: Full GridSearchCV (Computationally Expensive - Typically replaced by RandomizedSearch)
# from sklearn.model_selection import GridSearchCV, StratifiedKFold
# param_grid_full = CONFIG["xgb_param_grid"]
# cv_full = StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG["tuning_random_state"])
# grid_search_full = GridSearchCV(xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', tree_method='hist'),
#                            param_grid_full, scoring='roc_auc', cv=cv_full, n_jobs=-1, verbose=2)
# # This would be run on the X_sample, y_sample for comparison, or potentially full data if time permits
# # grid_search_full.fit(X_sample, y_sample) 
# logging.warning("Full GridSearchCV is commented out due to computational cost.")

In [ ]:
# Example Placeholder: Hyperparameter tuning for LightGBM 
# param_grid_lgb = {
#     'n_estimators': [100, 200],
#     'max_depth': [4, 6, 8],
#     # ... other LGBM params ...
# }
# grid_search_lgb = RandomizedSearchCV(lgb.LGBMClassifier(class_weight='balanced', random_state=CONFIG['lgb_random_state']),
#                            param_grid_lgb, scoring='roc_auc', cv=cv, n_jobs=-1, verbose=1, n_iter=CONFIG['tuning_n_iter'])
# # grid_search_lgb.fit(X_sample, y_sample)
# logging.warning("LightGBM tuning example is commented out.")

## Additional Potential Steps (Beyond Scope of Current Notebook)
- Model drift detection and monitoring infrastructure
- Model registry and versioning (e.g., using MLflow - example provided below)
- API deployment (FastAPI - example provided below)
- Frontend integration
- Documentation and reproducibility enhancements

### Example: Model Drift Detection & Monitoring (PSI - Basic Check)
Monitor model performance and detect data drift using Population Stability Index (PSI). The calculation was performed in Step 9b.

### Example: Model Registry & Versioning (MLflow)

In [ ]:
# Example: Register and log model using MLflow 
import mlflow
import mlflow.sklearn

# try:
#     # Set MLflow tracking URI (local or remote)
#     mlflow.set_tracking_uri("file:///path/to/your/mlruns") # Modify path as needed
#     mlflow.set_experiment("fraud_detection_model")
# 
#     # Register and log the chosen final model, parameters, metrics
#     with mlflow.start_run(run_name=f"final_{final_best_model_name.replace(' ', '_')}"):
#         mlflow.log_params(best_params) # Log best params found during tuning
#         # Log final metrics evaluated on the test set
#         for metric_name, metric_value in final_best_metrics.items():
#              if isinstance(metric_value, (int, float, np.number)):
#                   mlflow.log_metric(f"test_{metric_name}", metric_value)
# 
#         # Log other metadata
#         mlflow.log_param("chosen_model_type", final_best_model_name)
#         mlflow.log_param("smote_applied", True)
#         mlflow.log_param("tuning_iterations", CONFIG["tuning_n_iter"])
#         mlflow.log_dict(model_metadata, "model_metadata.json") # Log the metadata dict
# 
#         mlflow.sklearn.log_model(final_best_model, "model")
#         logging.info(f"Model logged in MLflow with run_id: {mlflow.active_run().info.run_id}")
# 
# except Exception as e:
#      logging.error(f"MLflow logging failed: {e}. Ensure MLflow is installed and configured.")

### Example: Model Deployment API (FastAPI Skeleton)

In [ ]:
# Example: FastAPI deployment skeleton (would be in a separate .py file)

# ```python
# from fastapi import FastAPI, Request
# import uvicorn
# import joblib
# import numpy as np
# import os
# 
# app = FastAPI()
# 
# # Determine model and scaler paths
# model_path = os.path.join(os.path.dirname(__file__), 'models', 'final_xgb_model.pkl')
# scaler_path = os.path.join(os.path.dirname(__file__), 'models', 'scaler.pkl')
# 
# try:
#     model = joblib.load(model_path)
#     scaler = joblib.load(scaler_path)
#     print(f"Model and scaler loaded successfully from {os.path.dirname(model_path)}")
# except FileNotFoundError:
#     print("Error: Model or scaler file not found. Ensure they are in the 'models' directory.")
#     # Handle error appropriately, e.g., exit or raise exception
#     model = None
#     scaler = None
# 
# @app.post("/predict")
# async def predict(request: Request):
#     if not model or not scaler:
#         return {"error": "Model or scaler not loaded"}, 500
# 
#     try:
#         data = await request.json()
#         # Assuming 'features' is a list or list of lists of numbers
#         features_raw = np.array(data["features"])
#         # Ensure it's 2D for the scaler
#         if features_raw.ndim == 1:
#             features_raw = features_raw.reshape(1, -1)
# 
#         # Apply the scaler
#         features_scaled = scaler.transform(features_raw)
# 
#         # Predict
#         preds = model.predict(features_scaled)
#         scores = model.predict_proba(features_scaled)[:, 1]
#         
#         # Return single prediction if single input, list otherwise
#         if features_raw.shape[0] == 1:
#              return {"prediction": int(preds[0]), "risk_score": float(scores[0])}
#         else:
#              return {"predictions": preds.tolist(), "risk_scores": scores.tolist()}
# 
#     except Exception as e:
#          return {"error": str(e)}, 400
# 
# # To run locally:
# # if __name__ == "__main__":
# #     uvicorn.run(app, host="0.0.0.0", port=8000)
# ```

### Note on Exporting Model for API Deployment
Step 10 already saves the final model (`final_xgb_model.pkl`) and scaler (`scaler.pkl`) to the specified `backend/models` directory. These are the files the example FastAPI app would load.

In [ ]:
# This cell is now redundant as Step 10 saves the final model and scaler.
# import os
# os.makedirs(CONFIG['models_dir'], exist_ok=True)
# joblib.dump(final_best_model, os.path.join(CONFIG['models_dir'], 'final_xgb_model.pkl'))
# joblib.dump(scaler, os.path.join(CONFIG['models_dir'], 'scaler.pkl'))
# logging.info('Final model and scaler exported again (redundant).')